In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

In [2]:
class BatsmanState(TypedDict):
    runs: int
    balls: int
    fours: int
    sixes: int

    sr: float
    bpb: float
    boundary_percentage: float
    detail_summary: str

In [3]:
graph = StateGraph(BatsmanState)

In [4]:
def calculate_sr(state: BatsmanState):
    sr = (state['runs'] / state['balls']) * 100 if state['balls'] > 0 else 0
    return {'sr': sr}

In [5]:
def calculate_bpb(state: BatsmanState):
    bpb = (state['balls']/(state['fours']+state['sixes'])) if (state['fours']+state['sixes'])>0 else 0
    return {'bpb': bpb}

In [6]:
def calculate_boundary_percentage(state: BatsmanState):
    boundary_percentage = ((state['fours']*4 + state['sixes']*6)/state['runs'])*100 if state['runs']>0 else 0
    return {'boundary_percentage': boundary_percentage}

In [7]:
def summary(state: BatsmanState):
    sumr = f"Runs: {state['runs']}, Balls: {state['balls']}, Fours: {state['fours']}, Sixes: {state['sixes']}, SR: {state['sr']:.2f}, BPB: {state['bpb']:.2f}, Boundary%: {state['boundary_percentage']:.2f}"
    return {'detail_summary': sumr}

In [8]:
#add nodes to the graph
graph.add_node("calculate_sr", calculate_sr)
graph.add_node("calculate_bpb", calculate_bpb)
graph.add_node("calculate_boundary_percentage", calculate_boundary_percentage)
graph.add_node("summary", summary)

#define edges to create parallel workflow
graph.add_edge(START, "calculate_sr")
graph.add_edge(START, "calculate_bpb")
graph.add_edge(START, "calculate_boundary_percentage")
graph.add_edge("calculate_sr", "summary")
graph.add_edge("calculate_bpb", "summary")
graph.add_edge("calculate_boundary_percentage", "summary")
graph.add_edge("summary", END)

workflow = graph.compile()

In [10]:
workflow.get_graph().print_ascii()

                                          +-----------+                                   
                                         *| __start__ |**                                 
                                    ***** +-----------+  *****                            
                              ******               *          ******                      
                         *****                     *                *****                 
                      ***                           *                    ***              
+-------------------------------+           +---------------+           +--------------+  
| calculate_boundary_percentage |           | calculate_bpb |           | calculate_sr |  
+-------------------------------+           +---------------+      *****+--------------+  
                              ******              *           *****                       
                                    *****         *     ******                            

In [15]:
initial_state: BatsmanState = {
    'runs': 120,
    'balls': 80,
    'fours': 10,
    'sixes': 5,
    'sr': 0.0,
    'bpb': 0.0,
    'boundary_percentage': 0.0,
    'summary': ''
}

In [20]:
final_state = workflow.invoke(initial_state)

In [21]:
final_state

{'runs': 120,
 'balls': 80,
 'fours': 10,
 'sixes': 5,
 'sr': 150.0,
 'bpb': 5.333333333333333,
 'boundary_percentage': 58.333333333333336,
 'detail_summary': 'Runs: 120, Balls: 80, Fours: 10, Sixes: 5, SR: 150.00, BPB: 5.33, Boundary%: 58.33'}